# №6 практикалық дәріс және зертханалық жұмыс
**Тақырып:** Деректерді қарапайым тазалау  
**Студент:** Шапағат  
**Топ:** 24-ПО-103-3к-5


In [ ]:
import pandas as pd
import numpy as np

print('Тазалау жұмысына дайын')

Тазалау жұмысына дайын


In [ ]:
csv_text = '''session_id,protocol,service,duration_sec,failed_logins,connections_1min,label
S001,TCP,http,12,0,4,normal
S002, tcp ,HTTP,18,1,7,Normal
S003,UDP,dns,unknown,2,9,normal
S004,TCP,http,42,3,18,attack
S005,ICMP,,60,5,25,Attack
S006,udp,DNS,15,,6,normal
S004,TCP,http,42,3,18,attack
S007,TCP,ssh,-5,4,21,attack
S008, tcp ,SSH,10,0,999,normal
S009,UDP,dns,22,1,8,normal
S010,ICMP,echo,55,4,24,attack
S011,TCP,http,30,2,12, normal
'''

with open('network_logs_06_dirty.csv', 'w', encoding='utf-8') as file:
    file.write(csv_text)

df = pd.read_csv('network_logs_06_dirty.csv')
df_original = df.copy(deep=True)
print('Бастапқы өлшем:', df.shape)
display(df)

Бастапқы өлшем: (12, 7)


,session_id,protocol,service,duration_sec,failed_logins,connections_1min,label
0,S001,TCP,http,12,0.0,4,normal
1,S002,tcp,HTTP,18,1.0,7,Normal
2,S003,UDP,dns,unknown,2.0,9,normal
3,S004,TCP,http,42,3.0,18,attack
4,S005,ICMP,NaN,60,5.0,25,Attack
5,S006,udp,DNS,15,NaN,6,normal
6,S004,TCP,http,42,3.0,18,attack
7,S007,TCP,ssh,-5,4.0,21,attack
8,S008,tcp,SSH,10,0.0,999,normal
9,S009,UDP,dns,22,1.0,8,normal


In [ ]:
print('Бос мәндер:')
print(df.isna().sum())
print('Барлық бос мән:', int(df.isna().sum().sum()))
print('Қайталанған жол:', int(df.duplicated().sum()))
print('Дерек түрлері:')
print(df.dtypes)
print('protocol мәндері:')
print(df['protocol'].value_counts(dropna=False))

Бос мәндер:
session_id          0
protocol            0
service             1
duration_sec        0
failed_logins       1
connections_1min    0
label               0
dtype: int64
Барлық бос мән: 2
Қайталанған жол: 1
Дерек түрлері:
session_id           object
protocol             object
service              object
duration_sec         object
failed_logins       float64
connections_1min      int64
label                object
dtype: object
protocol мәндері:
protocol
TCP      5
 tcp     2
UDP      2
ICMP     2
udp      1
Name: count, dtype: int64


In [ ]:
df_clean = df.copy(deep=True)

for column in ['protocol', 'service', 'label']:
    df_clean[column] = (
        df_clean[column].astype('string').str.strip().str.lower()
    )

print('Тазаланған protocol:')
print(df_clean['protocol'].value_counts(dropna=False))
print('Тазаланған service:')
print(df_clean['service'].value_counts(dropna=False))
print('Тазаланған label:')
print(df_clean['label'].value_counts(dropna=False))

Тазаланған protocol:
protocol
tcp     7
udp     3
icmp    2
Name: count, dtype: Int64
Тазаланған service:
service
http    5
dns     3
ssh     2
<NA>    1
echo    1
Name: count, dtype: Int64
Тазаланған label:
label
normal    7
attack    5
Name: count, dtype: Int64


In [ ]:
for column in ['duration_sec', 'failed_logins', 'connections_1min']:
    df_clean[column] = pd.to_numeric(df_clean[column], errors='coerce')

print('Сандық бағандардың түрлері:')
print(df_clean[['duration_sec', 'failed_logins', 'connections_1min']].dtypes)
print('Түрлендіруден кейінгі бос мәндер:')
print(df_clean.isna().sum())

Сандық бағандардың түрлері:
duration_sec        float64
failed_logins       float64
connections_1min      int64
dtype: object
Түрлендіруден кейінгі бос мәндер:
session_id          0
protocol            0
service             1
duration_sec        1
failed_logins       1
connections_1min    0
label               0
dtype: int64


In [ ]:
negative_duration = int((df_clean['duration_sec'] < 0).sum())
print('Теріс duration_sec саны:', negative_duration)
df_clean.loc[df_clean['duration_sec'] < 0, 'duration_sec'] = np.nan

duration_median = df_clean['duration_sec'].median()
failed_median = df_clean['failed_logins'].median()
df_clean['duration_sec'] = df_clean['duration_sec'].fillna(duration_median)
df_clean['failed_logins'] = df_clean['failed_logins'].fillna(failed_median)

service_mode = df_clean['service'].mode().iloc[0]
df_clean['service'] = df_clean['service'].fillna(service_mode)
print('Медианалар:', duration_median, failed_median)
print('Ең жиі service:', service_mode)
print('Қалған бос мән:', int(df_clean.isna().sum().sum()))

Теріс duration_sec саны: 1
Медианалар: 26.0 2.0
Ең жиі service: http
Қалған бос мән: 0


In [ ]:
before_rows = len(df_clean)
df_clean = df_clean.drop_duplicates().copy()
print('Жойылған қайталанған жол:', before_rows - len(df_clean))
print('Бұрынғы connections_1min максимумы:', df_clean['connections_1min'].max())
df_clean['connections_1min'] = df_clean['connections_1min'].clip(lower=0, upper=300)
print('Жаңа максимум:', df_clean['connections_1min'].max())

Жойылған қайталанған жол: 1
Бұрынғы connections_1min максимумы: 999
Жаңа максимум: 300


In [ ]:
assert df_clean.shape == (11, 7)
assert df_clean.isna().sum().sum() == 0
assert df_clean.duplicated().sum() == 0
assert (df_clean['duration_sec'] >= 0).all()
assert df_clean['connections_1min'].max() <= 300
assert set(df_clean['protocol']) <= {'tcp', 'udp', 'icmp'}
assert df.equals(df_original)

print('Соңғы өлшем:', df_clean.shape)
print('Бос мәндер:')
print(df_clean.isna().sum())
print('Қайталану:', int(df_clean.duplicated().sum()))
print('Түрлер:')
print(df_clean.dtypes)
print('Барлық тексеру сәтті өтті')
display(df_clean)

Соңғы өлшем: (11, 7)
Бос мәндер:
session_id          0
protocol            0
service             0
duration_sec        0
failed_logins       0
connections_1min    0
label               0
dtype: int64
Қайталану: 0
Түрлер:
session_id                  object
protocol            string[python]
service             string[python]
duration_sec               float64
failed_logins              float64
connections_1min             int64
label               string[python]
dtype: object
Барлық тексеру сәтті өтті


,session_id,protocol,service,duration_sec,failed_logins,connections_1min,label
0,S001,tcp,http,12.0,0.0,4,normal
1,S002,tcp,http,18.0,1.0,7,normal
2,S003,udp,dns,26.0,2.0,9,normal
3,S004,tcp,http,42.0,3.0,18,attack
4,S005,icmp,http,60.0,5.0,25,attack
5,S006,udp,dns,15.0,2.0,6,normal
7,S007,tcp,ssh,26.0,4.0,21,attack
8,S008,tcp,ssh,10.0,0.0,300,normal
9,S009,udp,dns,22.0,1.0,8,normal
10,S010,icmp,echo,55.0,4.0,24,attack


In [ ]:
comparison = pd.DataFrame({
    'көрсеткіш': ['жол саны', 'бос мән саны', 'қайталанған жол'],
    'бастапқы': [len(df_original), int(df_original.isna().sum().sum()),
                 int(df_original.duplicated().sum())],
    'таза': [len(df_clean), int(df_clean.isna().sum().sum()),
             int(df_clean.duplicated().sum())]
})
display(comparison)

df_clean.to_csv('network_logs_06_clean.csv', index=False, encoding='utf-8')
print('Сақталды: network_logs_06_clean.csv')

,көрсеткіш,бастапқы,таза
0,жол саны,12,11
1,бос мән саны,2,0
2,қайталанған жол,1,0


Сақталды: network_logs_06_clean.csv


In [ ]:
print('1. Барлық баған:', df.isna().sum().to_dict())
print('2. service бос:', df['service'].isna().sum())
print('3. Қайталану саны:', df.duplicated().sum())
print('4. Қайталанған жол:')
display(df[df.duplicated()])
df_test = df.copy()
print('5. Көшірменің өлшемі:', df_test.shape)

df_test['protocol'] = df_test['protocol'].astype('string').str.strip()
print('6. protocol:', df_test['protocol'].unique())
df_test['service'] = df_test['service'].astype('string').str.lower()
print('7. service:', df_test['service'].unique())
df_test['duration_sec'] = pd.to_numeric(df_test['duration_sec'], errors='coerce')
print('8. duration_sec түрі:', df_test['duration_sec'].dtype)

print('9. Теріс уақыт:', (df_test['duration_sec'] < 0).sum())
df_test.loc[df_test['duration_sec'] < 0, 'duration_sec'] = np.nan
print('10. Теріс уақыт қалды:', (df_test['duration_sec'] < 0).sum())

df_test['failed_logins'] = pd.to_numeric(df_test['failed_logins'], errors='coerce')
df_test['failed_logins'] = df_test['failed_logins'].fillna(df_test['failed_logins'].median())
print('11. failed_logins бос:', df_test['failed_logins'].isna().sum())
df_test['service'] = df_test['service'].fillna(df_test['service'].mode().iloc[0])
print('12. service бос:', df_test['service'].isna().sum())

df_test = df_test.drop_duplicates().copy()
print('13. Қайталану қалды:', df_test.duplicated().sum())
df_test['connections_1min'] = pd.to_numeric(df_test['connections_1min'], errors='coerce').clip(0, 300)
print('14. Ең үлкен байланыс саны:', df_test['connections_1min'].max())
df_clean.to_csv('my_clean_data.csv', index=False)
print('15. Сақталды: my_clean_data.csv')

1. Барлық баған: {'session_id': 0, 'protocol': 0, 'service': 1, 'duration_sec': 0, 'failed_logins': 1, 'connections_1min': 0, 'label': 0}
2. service бос: 1
3. Қайталану саны: 1
4. Қайталанған жол:


,session_id,protocol,service,duration_sec,failed_logins,connections_1min,label
6,S004,TCP,http,42,3.0,18,attack


5. Көшірменің өлшемі: (12, 7)
6. protocol: <StringArray>
['TCP', 'tcp', 'UDP', 'ICMP', 'udp']
Length: 5, dtype: string
7. service: <StringArray>
['http', 'dns', <NA>, 'ssh', 'echo']
Length: 5, dtype: string
8. duration_sec түрі: float64
9. Теріс уақыт: 1
10. Теріс уақыт қалды: 0
11. failed_logins бос: 0
12. service бос: 0
13. Қайталану қалды: 0
14. Ең үлкен байланыс саны: 300
15. Сақталды: my_clean_data.csv


In [ ]:
wrong_example = pd.DataFrame({'protocol': [' TCP ', 'Udp']})
wrong_example['protocol'].str.strip().str.lower()
print('Қате нұсқадан кейін:', wrong_example['protocol'].tolist())

wrong_example['protocol'] = wrong_example['protocol'].str.strip().str.lower()
print('Түзетілген нұсқа:', wrong_example['protocol'].tolist())

Қате нұсқадан кейін: [' TCP ', 'Udp']
Түзетілген нұсқа: ['tcp', 'udp']


In [ ]:
student_data = pd.DataFrame({
    'event_id': [1, 2, 3, 3, 4, 5, 6, 7],
    'protocol': [' TCP ', 'udp', 'UDP ', 'udp', 'ICMP', None, 'tcp', 'TCP'],
    'packets': ['12', '18', 'bad', '18', '25', None, '-4', '500'],
    'status': ['Normal', 'normal ', 'Attack', 'Attack', 'normal', 'Normal', 'attack ', 'NORMAL']
})
student_original = student_data.copy(deep=True)
print('Бастапқы өлшем:', student_data.shape)
print('Бастапқы бос мәндер:', student_data.isna().sum().to_dict())
print('Бастапқы қайталану:', int(student_data.duplicated().sum()))
print('Бастапқы түрлер:', student_data.dtypes.to_dict())
display(student_data)

Бастапқы өлшем: (8, 4)
Бастапқы бос мәндер: {'event_id': 0, 'protocol': 1, 'packets': 1, 'status': 0}
Бастапқы қайталану: 0
Бастапқы түрлер: {'event_id': dtype('int64'), 'protocol': dtype('O'), 'packets': dtype('O'), 'status': dtype('O')}


,event_id,protocol,packets,status
0,1,TCP,12,Normal
1,2,udp,18,normal
2,3,UDP,bad,Attack
3,3,udp,18,Attack
4,4,ICMP,25,normal
5,5,None,None,Normal
6,6,tcp,-4,attack
7,7,TCP,500,NORMAL


In [ ]:
student_clean = student_data.copy(deep=True)
for column in ['protocol', 'status']:
    student_clean[column] = student_clean[column].astype('string').str.strip().str.lower()

student_clean['packets'] = pd.to_numeric(student_clean['packets'], errors='coerce')
student_clean.loc[student_clean['packets'] < 0, 'packets'] = np.nan
student_clean['packets'] = student_clean['packets'].fillna(student_clean['packets'].median())
student_clean['protocol'] = student_clean['protocol'].fillna(
    student_clean['protocol'].mode().iloc[0]
)

before_student_rows = len(student_clean)
student_clean = student_clean.drop_duplicates().copy()
student_clean['packets'] = student_clean['packets'].clip(0, 300)

assert student_clean.shape == (7, 4)
assert student_clean.isna().sum().sum() == 0
assert student_clean.duplicated().sum() == 0
assert student_original.equals(student_data)

print('Жойылған қайталанған жол:', before_student_rows - len(student_clean))
print('Соңғы бос мәндер:', student_clean.isna().sum().to_dict())
print('Соңғы қайталану:', int(student_clean.duplicated().sum()))
print('protocol:')
print(student_clean['protocol'].value_counts())
print('status:')
print(student_clean['status'].value_counts())
display(student_clean)
student_clean.to_csv('student_network_clean.csv', index=False, encoding='utf-8')
print('Сақталды: student_network_clean.csv')

Жойылған қайталанған жол: 1
Соңғы бос мәндер: {'event_id': 0, 'protocol': 0, 'packets': 0, 'status': 0}
Соңғы қайталану: 0
protocol:
protocol
tcp     4
udp     2
icmp    1
Name: count, dtype: Int64
status:
status
normal    5
attack    2
Name: count, dtype: Int64


,event_id,protocol,packets,status
0,1,tcp,12.0,normal
1,2,udp,18.0,normal
2,3,udp,18.0,attack
4,4,icmp,25.0,normal
5,5,tcp,18.0,normal
6,6,tcp,18.0,attack
7,7,tcp,300.0,normal


Сақталды: student_network_clean.csv
